# Bronze Table Creation: Customer Churn Data

This notebook loads customer churn data from a Unity Catalog volume and creates a bronze layer table.

**Source**: `vol_demo.dw_raw.customer_churn` volume  
**Destination**: `crm.customer_info.customer_churn_bronze` table  
**Format**: Delta Lake (managed table)

---

In [0]:
# List files in the customer_churn volume
volume_path = "/Volumes/vol_demo/dw_raw/customer_churn"
files = dbutils.fs.ls(volume_path)

print("Files in volume:")
for file in files:
    print(f"  Name: {file.name}")
    print(f"  Size: {file.size} bytes")
    print()

In [0]:
# Read and preview the CSV file to understand structure
file_path = "/Volumes/vol_demo/dw_raw/customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)

print(f"Total rows: {df.count()}")
print(f"\nSchema:")
df.printSchema()
print(f"\nSample data:")
df.show(5, truncate=False)

In [0]:
# Check if the target schema (crm.customer_info) exists
# First, verify current catalog
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
print(f"Current catalog: {current_catalog}")

# List all available catalogs
catalogs = spark.sql("SHOW CATALOGS").collect()
print("\nAvailable catalogs:")
for cat in catalogs:
    print(f"  - {cat.catalog}")

# Check if 'crm' catalog exists and list its schemas
try:
    schemas = spark.sql("SHOW SCHEMAS IN crm").collect()
    schema_names = [row.databaseName for row in schemas]
    print(f"\nSchemas in 'crm' catalog: {schema_names}")
    
    if 'customer_info' in schema_names:
        print("\n✓ Target schema 'crm.customer_info' exists")
    else:
        print("\n✗ Schema 'customer_info' does not exist in 'crm' catalog")
except Exception as e:
    print(f"\nError: {e}")

In [0]:
# Read the data from volume
file_path = "/Volumes/vol_demo/dw_raw/customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)

# Define target table name
table_name = "crm.customer_info.customer_churn_bronze"

# Write as a managed Delta table in the bronze layer
# Mode 'overwrite' will replace the table if it already exists
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable(table_name)

print(f"✓ Successfully created bronze table: {table_name}")
print(f"✓ Total records loaded: {df.count()}")

# Display table metadata
print(f"\nTable schema:")
spark.sql(f"DESCRIBE TABLE {table_name}").show(truncate=False)

In [0]:
# Query the bronze table to verify data and get initial insights
sample_query = """
SELECT 
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate_pct
FROM crm.customer_info.customer_churn_bronze
"""

print("Summary statistics:")
spark.sql(sample_query).show()

print("\nSample records from bronze table:")
spark.sql("SELECT * FROM crm.customer_info.customer_churn_bronze LIMIT 3").show(truncate=False)

## Summary

Bronze table successfully created with the following characteristics:

* **Table**: `crm.customer_info.customer_churn_bronze`
* **Records**: 7,043 customer records
* **Columns**: 21 columns including:
  - Customer demographics (gender, SeniorCitizen, Partner, Dependents)
  - Service details (PhoneService, InternetService, MultipleLines, etc.)
  - Account information (tenure, Contract, PaymentMethod)
  - Financial data (MonthlyCharges, TotalCharges)
  - Target variable (Churn)
* **Format**: Delta Lake managed table
* **Churn Rate**: ~26.54% (1,869 out of 7,043 customers)

### Next Steps
1. Create silver layer tables with data quality checks and transformations
2. Build gold layer aggregated tables for analytics
3. Develop churn prediction models